# 01 — Data audit

Dataset discovery and schema inspection, data-quality statistics, and the frozen analysis dataset (Req 7.1). Raw CEDA files must be present in `data/raw/` (see `data/raw/README.md`).

*Data: CEDA Agri Market Data, Centre for Economic Data and Analysis (CEDA), Ashoka University; original source DMI, Ministry of Agriculture and Farmers Welfare (Agmarknet).*

In [1]:
import os, sys
from pathlib import Path
# Run from the repository root so `src` and configured relative paths resolve
root = Path.cwd()
while not (root / "configs" / "config.json").exists() and root != root.parent:
    root = root.parent
os.chdir(root)
sys.path.insert(0, str(root))
import json
import pandas as pd
from IPython.display import Image, Markdown, display
from src.config import load_config, resolve_path
cfg = load_config()
tables = resolve_path(cfg["paths"]["tables_dir"])
figs = resolve_path(cfg["paths"]["figures_dir"])
print("Repository root:", root)

Repository root: D:\MY works\CropCast


## Offline integrity audit of the 56 raw chunks
`src.data.validate.run_audit` checks the manifest, checksums, schemas, duplicates, nulls, numeric rules, coverage and pilot overlap.

In [2]:
from src.data.validate import run_audit
audit = run_audit(manifest_path=resolve_path(cfg['paths']['manifest']), raw_dir=resolve_path(cfg['paths']['raw_dir']))
print('passed:', audit['passed']); print('hard failures:', audit['hard_failures']); print('warnings:', audit['warnings'])
pd.DataFrame({sid: {k: m[k] for k in ['chunks_found','price_records_count','quantity_records_count','duplicate_price_keys','null_values','numeric_violations','modal_bounds_warnings','missing_price_days','price_gaps_count','max_consecutive_price_gap','iou']} for sid, m in audit['series_metrics'].items()})

passed: True
hard failures: []
warnings: ['onion_maharashtra: 10 values beyond 100x of the field median', 'potato_uttar_pradesh: 137 records with modal_price outside [min_price, max_price]']


,onion_maharashtra,potato_uttar_pradesh
chunks_found,28.0,28.0
price_records_count,2523.0,2557.0
quantity_records_count,2523.0,2557.0
duplicate_price_keys,0.0,0.0
null_values,0.0,0.0
numeric_violations,0.0,0.0
modal_bounds_warnings,0.0,137.0
missing_price_days,34.0,0.0
price_gaps_count,13.0,0.0
max_consecutive_price_gap,10.0,0.0


## Build the frozen analysis dataset (v1)
Deterministic and lossless; the CSV checksum must match `data/processed/cropcast_analysis_v1.meta.json`.

In [3]:
from src.data.dataset import build_dataset, write_dataset
df, meta = build_dataset(cfg)
csv_path, meta_path = write_dataset(df, meta, resolve_path(cfg['paths']['processed_dir']), cfg['dataset']['version'])
print(json.loads(meta_path.read_text())['csv_sha256'])
pd.DataFrame(meta['missing_report'])

b5f21e02b681c7887af340ba6e5d22ec9b3847bfd171da8e7ba3ffc24152b773


,series_id,commodity,market,calendar_days,observed_days,missing_days,missing_pct
0,onion_maharashtra,Onion,Maharashtra (state aggregate),2557,2523,34,1.33
1,potato_uttar_pradesh,Potato,Uttar Pradesh (state aggregate),2557,2557,0,0.00


In [4]:
pd.DataFrame(meta['flag_counts']).T

,flag_modal_outside_bounds,flag_extreme_arrival
onion_maharashtra,0,10
potato_uttar_pradesh,137,0


In [5]:
df.groupby('series_id')[['modal_price','min_price','max_price','arrival_qty']].describe().T

series_id          onion_maharashtra  potato_uttar_pradesh
modal_price count        2523.000000           2557.000000
            mean         1388.292105           1016.849541
            std           963.195681            493.453910
            min           406.163265            433.557692
            25%           710.033333            646.341463
            50%          1079.254902            952.073171
            75%          1755.568966           1230.340782
            max          7506.450980           3015.180723
min_price   count        2523.000000           2557.000000
            mean          706.700535            967.550885
            std           449.484660            484.755234
            min           150.000000            405.000000
            25%           377.423913            597.904192
            50%           572.560976            897.045455
            75%           865.875000           1173.064516
            max          3171.782609           2928.704819
max_price   count        2523.000000           2557.000000
            mean         1801.100042           1078.886460
            std          1288.000306            501.941311
            min           568.785714            484.411765
            25%           920.780676            702.373057
            50%          1399.861111           1012.125000
            75%          2292.056369           1295.903614
            max         10913.175000           3098.515152
arrival_qty count        2523.000000           2557.000000
            mean        16323.068173          15363.745999
            std          9001.509583           8593.779063
            min             2.000000            442.000000
            25%          9069.000000           9250.830000
            50%         15664.000000          14683.250000
            75%         22739.000000          19919.970000
            max         45648.000000         113475.300000